In [1]:
#add libraries
%pip install pandas matplotlib numpy
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from read_data import save_files

Note: you may need to restart the kernel to use updated packages.


### Read Data

In [2]:
def read_data():
    return [
        pd.read_csv('../data/01-starting_data/development_data/awards_players.csv'),
        pd.read_csv('../data/01-starting_data/development_data/coaches.csv'),
        pd.read_csv('../data/01-starting_data/development_data/players.csv'),
        pd.read_csv('../data/01-starting_data/development_data/players_teams.csv'),
        pd.read_csv('../data/01-starting_data/development_data/series_post.csv'),
        pd.read_csv('../data/01-starting_data/development_data/teams.csv'),
        pd.read_csv('../data/01-starting_data/development_data/teams_post.csv')
    ]

awards_players, coaches, players, players_teams, series_post, teams, teams_post = read_data()

### Data Selection

Section where we select relevant data and filter out invariant or irrelevant columns 

In [3]:
awards_players = awards_players.drop(columns=['lgID']) 
coaches = coaches.drop(columns=['lgID'])
players = players.drop(columns=['firstseason', 'lastseason', 'college', 'collegeOther', 'deathDate'])
players_teams = players_teams.drop(columns=['lgID'])
series_post = series_post.drop(columns=['lgIDWinner', 'lgIDLoser']) #'round', 'series' too ??
teams = teams.drop(columns=['lgID', 'franchID', 'divID', 'arena', 'name', 'seeded'])
teams_post = teams_post.drop(columns=['lgID'])

In [4]:
save_files("02-data_selection", 
           ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"], 
           [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

Data awards_players saved to ../data/02-data_selection
Data coaches saved to ../data/02-data_selection
Data players saved to ../data/02-data_selection
Data players_teams saved to ../data/02-data_selection
Data series_post saved to ../data/02-data_selection
Data teams saved to ../data/02-data_selection
Data teams_post saved to ../data/02-data_selection


### Data Preparation

Section where we treat cases like non-existing values, outliers, etc.

In [5]:
save_files("03-data_preparation",
           ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
           [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

Data awards_players saved to ../data/03-data_preparation
Data coaches saved to ../data/03-data_preparation
Data players saved to ../data/03-data_preparation
Data players_teams saved to ../data/03-data_preparation
Data series_post saved to ../data/03-data_preparation
Data teams saved to ../data/03-data_preparation
Data teams_post saved to ../data/03-data_preparation


### Feature Engineering

Section where we create and/or simplify existing metrics to aid the prediction model

In [6]:
# def calculateUPER():
#     data['uPER'] = 

# def calculatePER():
#     data['PER'] = uPER * (lgPace/tmPace) * (15 / lguPER) 

In [7]:
teams['win_loss_ratio'] = teams['won'] / (teams['won'] + teams['lost'])  
#data['next_year_playoff_qualification'] = data.groupby('tmID')['playoff_qualification'].shift(-1)

columns_to_drop = ['lgID', 'rank', 'firstRound', 'semis', 'finals']
teams = teams.drop(columns=columns_to_drop, errors='ignore')

#data.columns

In [8]:
save_files("04-feature_engineering",
            ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
            [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

Data awards_players saved to ../data/04-feature_engineering
Data coaches saved to ../data/04-feature_engineering
Data players saved to ../data/04-feature_engineering
Data players_teams saved to ../data/04-feature_engineering
Data series_post saved to ../data/04-feature_engineering
Data teams saved to ../data/04-feature_engineering
Data teams_post saved to ../data/04-feature_engineering


### Data Merging

Section responsible for merging all tables, in a format ready to feed the model

In [9]:
#team metrics

data = pd.merge(teams, teams_post, on=['year', 'tmID'], how='left')

data['playoff_qualification'] = data['playoff'].apply(lambda x: 1.0 if x == 'Y' else 0.0)
data.drop(columns=['playoff'], inplace=True)
data.fillna({'W' : 0, 'L' : 0}, inplace=True)
    
#player stats
player_stats = players_teams.groupby(['tmID', 'year']).agg({
    'points': 'sum',
    'rebounds': 'sum',
    'assists': 'sum',
    'steals': 'sum',
    'blocks': 'sum',
    'turnovers': 'sum'
}).reset_index()

data = pd.merge(data, player_stats, on=['year', 'tmID'], how='left')

coach_stats = coaches.groupby(['year', 'tmID']).agg({
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()

data = pd.merge(data, coach_stats, on=["year", "tmID"], how="left")

data.columns
#awards
# awards_count = awards_players.groupby(['playerID', 'year']).size().reset_index(name='num_awards')

# data = pd.merge(data, awards_count, on=["year", "playerID"], how="left")
# data['num_awards'] = data['num_awards'].fillna(0)

Index(['year', 'tmID', 'confID', 'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm',
       'o_3pa', 'o_oreb', 'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to',
       'o_blk', 'o_pts', 'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa',
       'd_oreb', 'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk',
       'd_pts', 'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB',
       'won_x', 'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW',
       'confL', 'min', 'attend', 'win_loss_ratio', 'W', 'L',
       'playoff_qualification', 'points', 'rebounds', 'assists', 'steals',
       'blocks', 'turnovers', 'won_y', 'lost_y', 'post_wins', 'post_losses'],
      dtype='object')

### Add Year 11 info

In [10]:
def read_data11():
    return [
        pd.read_csv('../data/01-starting_data/challenge/coaches.csv'),
        pd.read_csv('../data/01-starting_data/challenge/players_teams.csv'),
        pd.read_csv('../data/01-starting_data/challenge/teams.csv')
    ]

coaches11, players_teams11, teams11 = read_data11()

coaches11 = coaches11.drop(columns=['lgID'])
players_teams11 = players_teams11.drop(columns=['lgID'])
teams11 = teams11.drop(columns=['lgID', 'franchID', 'arena', 'name'])

# #data11 = pd.merge(teams11, players_teams11, on=['year', 'tmID'], how='left')
# #data11 = pd.merge(data11, coaches11, on=['year', 'tmID'], how='left')

data = pd.concat([data, teams11], ignore_index=True)


In [11]:
def shift_performance(data, columns):
    """
    Shifts the performance metrics for next year
    """
    for column in columns:
        data = data.assign(**{column: data.groupby('tmID')[column].shift(1)})
    return data

data = shift_performance(data, [column for column in data.columns if column not in ['year', 'tmID', 'confID', 'playoff_qualification']])
data.fillna(0, inplace=True)


In [12]:
#write df to csv
save_files("05-processed_data", ["processed_data"], [data])

data.fillna(0, inplace=True)

print(f"final data columns {data.columns}")

Data processed_data saved to ../data/05-processed_data
final data columns Index(['year', 'tmID', 'confID', 'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm',
       'o_3pa', 'o_oreb', 'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to',
       'o_blk', 'o_pts', 'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa',
       'd_oreb', 'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk',
       'd_pts', 'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB',
       'won_x', 'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW',
       'confL', 'min', 'attend', 'win_loss_ratio', 'W', 'L',
       'playoff_qualification', 'points', 'rebounds', 'assists', 'steals',
       'blocks', 'turnovers', 'won_y', 'lost_y', 'post_wins', 'post_losses'],
      dtype='object')


### Model training

In [13]:
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVR

Note: you may need to restart the kernel to use updated packages.


#### Initialization 

In [14]:
#result lists
accuracy_scores = []
error_scores = []

#separate feature and target columns
feature_columns = [col for col in data.columns if col not in ['playoff_qualification', 'tmID', 'confID']]
target_column = 'playoff_qualification'

#Create model
#model = DecisionTreeClassifier(random_state=21) #acc: 0.55 error: 0.59 || acc: 0.59  error: 0.55
model = RandomForestClassifier(random_state=21) #acc: 0.60 error: 0.43 || acc: 0.64  error: 0.44
#model = LogisticRegression(random_state=21)     #acc: 0.64 error: 0.39 || acc: 0.57  error: 0.44
#model = SVR()                                   #acc: 0.62 error: 0.44 || acc: 0.60  error: 0.45

data.columns

Index(['year', 'tmID', 'confID', 'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm',
       'o_3pa', 'o_oreb', 'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to',
       'o_blk', 'o_pts', 'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa',
       'd_oreb', 'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk',
       'd_pts', 'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB',
       'won_x', 'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW',
       'confL', 'min', 'attend', 'win_loss_ratio', 'W', 'L',
       'playoff_qualification', 'points', 'rebounds', 'assists', 'steals',
       'blocks', 'turnovers', 'won_y', 'lost_y', 'post_wins', 'post_losses'],
      dtype='object')

#### Year training cycle

In [15]:
for year in sorted(data['year'].unique())[1:]:  # Start from the second year (with )
    # Separate train and test data
    year_span = 3
    train_data = data[data['year'] <= year] if year < (year_span - 1) else data[(data['year'] <= year) & (data['year'] >= year - (year_span - 1))]
    test_data = data[data['year'] == year + 1]
    
    # If there's no data for the next year (e.g., last year in the dataset), skip
    if test_data.empty:
        continue

    confIDs = test_data['confID'].values
    teamIDs = test_data['tmID'].values

    # Split features and target
    X_train = train_data[feature_columns]
    y_train = train_data[target_column]

    X_test = test_data[feature_columns]
    y_test = test_data[target_column]
    
    # Standardize the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred_proba = model.predict_proba(X_test)[:,1]
    y_pred_proba_norm = np.round(y_pred_proba * 8 / sum(y_pred_proba),2)  # Normalize to sum to 8

    #results_df['Playoff'] = results_df['Playoff'].apply(lambda x: ) # normalize -> max = 1?

    results_df = pd.DataFrame({
        'tmID': teamIDs,
        'confID': confIDs,
        'Playoff': np.round(y_pred_proba_norm, 2),
    })

    y_pred = np.zeros_like(y_pred_proba_norm)

    for conf_id in results_df['confID'].unique():
        conf_group = results_df[results_df['confID'] == conf_id]
        top_4_indices = conf_group.nlargest(4, 'Playoff').index
        y_pred[results_df.index.isin(top_4_indices)] = 1

    # indices = np.argsort(y_pred_proba_norm)[-8:]  # Get indices of top 8 probabilities
    # y_pred[indices] = 1                      # and set them to 1

    results_df['Label'] = y_pred

    if (year == 10):
        results_df = results_df.drop(columns=['Label', 'confID'])
        results_df.to_csv('../data/06-results/results.csv', index=False)
    # Calculate accuracy and error

    if year < 10:
        accuracy_scores.append(round(accuracy_score(y_test, y_pred),2))

        error_array = np.abs(y_pred_proba_norm - y_test.values)
        error_score = round(sum(error_array) / len(error_array),2)
        error_scores.append(error_score)

        # Output results for each year
        print(f"Year {year} -> {year + 1}:")
        print(f"Results: \n predict: \n {results_df}\n label: \t {y_pred}\n expected: {y_test.values}\n error: \t {error_array}")
        print(f"  Accuracy: {accuracy_scores[-1]}")
        print(f"  Error: \t {round(sum(error_array), 2)} / {len(error_array)} ({error_score})")
        print("\n")

Year 2 -> 3:
Results: 
 predict: 
    tmID confID  Playoff  Label
0   CHA     EA     0.55    1.0
1   CLE     EA     0.72    1.0
2   DET     EA     0.40    0.0
3   HOU     WE     0.53    1.0
4   IND     EA     0.27    0.0
5   LAS     WE     0.88    1.0
6   MIA     EA     0.58    1.0
7   MIN     WE     0.34    0.0
8   NYL     EA     0.74    1.0
9   ORL     EA     0.37    0.0
10  PHO     WE     0.34    0.0
11  POR     WE     0.24    0.0
12  SAC     WE     0.81    1.0
13  SEA     WE     0.32    0.0
14  UTA     WE     0.48    1.0
15  WAS     EA     0.40    0.0
 label: 	 [1. 1. 0. 1. 0. 1. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0.]
 expected: [1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 1.]
 error: 	 [0.45 0.72 0.4  0.47 0.73 0.12 0.58 0.34 0.26 0.37 0.34 0.24 0.81 0.68
 0.52 0.6 ]
  Accuracy: 0.62
  Error: 	 7.63 / 16 (0.48)


Year 3 -> 4:
Results: 
 predict: 
    tmID confID  Playoff  Label
0   CHA     EA     0.67    1.0
1   CLE     EA     0.52    0.0
2   CON     EA     0.57    1.0
3   DET     EA    

### End Results

In [16]:
print(f"Accuracy  {accuracy_scores}")
print(f"Error \t {[float(e) for e in error_scores]}")
print("\nAverage Performance Over All Years:")
print(f"  Average Accuracy: {sum(accuracy_scores) / len(accuracy_scores):.2f}")
print(f"  Average Error: \t {round(sum(error_scores) / len(error_scores),2)}")

Accuracy  [0.62, 0.43, 0.54, 0.69, 0.57, 0.54, 0.71, 0.54]
Error 	 [0.48, 0.49, 0.43, 0.43, 0.43, 0.44, 0.41, 0.47]

Average Performance Over All Years:
  Average Accuracy: 0.58
  Average Error: 	 0.45
